# 🧠 NV Gamer — 超汎用・30×人間味・マルチゲーム基盤AI ワンクリック GPU 強化学習

> ※ これは **NVSS (`NVSS_Train_Colab_Pro.ipynb`) とは完全独立した NV Gamer 専用ノートブック** です。

**NV Gamer** は、特定のゲームに依存しない **汎用エンティティTransformer + Meta-RL GRU作業記憶 + 自己教師あり順方向ワールドモデル + 仮想人間コントローラー（曲線Minimum-Jerkマウス＆キーボード指慣性）** を統合した革新的アーキテクチャです。

### このノートブックで鍛えられる5つの異種ゲームジャンル（同時GPU並列）
1. **Tactical 3D FPS**（遮蔽・天候弾道歪み・ドローン・ガス管罠・対空戦）
2. **Arcade High-Speed Racing**（ステアリング・コーナー進入・ドリフト・ニトロ・障害物回避）
3. **Bullet-Hell / Action Platformer**（高密度弾幕の垂直回避・無敵ダッシュ・空中制御）
4. **MOBA / Survival Skirmish**（引き撃ち・資源回収・回復/AOEクールダウン管理）
5. **Unknown Adaptive Game**（試合中に操作軸反転・物理変化・未知のボタンが突然発生する超適応訓練）

### ワンクリック実行手順
1. **ランタイム → ランタイムのタイプを変更 → GPU（A100 / L4 推奨）** を選択。
2. **ランタイム → すべてのセルを実行 (`Ctrl+F9`)** をクリック！
3. 自動ダウンロードされる **`nvgamer-weights.json`** は **Result** の **「NV Gamer読込」** ボタンはもちろん、**他の自作ゲーム** に `dist/js/nvgamer-core.js` と一緒にポン付けして動かせます！

In [ ]:
#@title ⚙️ ステップ1: 環境セットアップ・Colab Pro GPU自動検出・Google Drive自動保存 { display-mode: "form" }
REPO_URL = "https://github.com/sc8z35a-collab/Result.git" #@param {type:"string"}
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_SAVE_DIR = "/content/drive/MyDrive/NV_Gamer_Colab_Pro_Runs" #@param {type:"string"}

import os, subprocess, torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ Colab Pro GPU 検出: {gpu} ({vram:.1f} GB VRAM) | bf16={torch.cuda.is_bf16_supported()}")
else:
    print("⚠️ GPU が無効です。ランタイム → ランタイムのタイプを変更 → GPU (A100/L4) を選択してください！")

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUT_DIR = DRIVE_SAVE_DIR
    except Exception as e:
        print(f"Drive マウントをスキップ ({e}) -> ローカル保存を使用")
        OUT_DIR = "/content/nvgamer_runs"
else:
    OUT_DIR = "/content/nvgamer_runs"

os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists("/content/Result"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/Result"], check=True)
else:
    subprocess.run(["git", "-C", "/content/Result", "pull", "--ff-only"], check=False)

print(f"✅ NV Gamer セットアップ完了！保存先: {OUT_DIR}")

In [ ]:
#@title 🧠 ステップ2: NV Gamer ワンクリック Multi-Game GPU 強化学習スタート { display-mode: "form" }
PRESET = "Pro: 2億ステップ (5ジャンル汎用・A100で約15〜25分)" #@param ["Quick: 3,000万ステップ (お試し約3分)", "Pro: 2億ステップ (5ジャンル汎用・A100で約15〜25分)", "Ultra: 10億ステップ (Drive自動再開で超人級汎用化)", "Billion+: 50億ステップ (究極の汎用ゲーマー基盤モデル)"]
RESET_CHECKPOINT = False #@param {type:"boolean"}

preset_map = {
    "Quick: 3,000万ステップ (お試し約3分)": 30_000_000,
    "Pro: 2億ステップ (5ジャンル汎用・A100で約15〜25分)": 200_000_000,
    "Ultra: 10億ステップ (Drive自動再開で超人級汎用化)": 1_000_000_000,
    "Billion+: 50億ステップ (究極の汎用ゲーマー基盤モデル)": 5_000_000_000,
}
TARGET_STEPS = preset_map[PRESET]

cmd = [
    "python3", "/content/Result/colab/train_nvgamer.py",
    "--target-steps", str(TARGET_STEPS),
    "--out-dir", OUT_DIR,
    "--log-every", "5",
    "--save-every", "20",
]
if RESET_CHECKPOINT:
    cmd.append("--fresh")

print("実行コマンド:", " ".join(cmd))
import subprocess, sys
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
rc = proc.wait()
assert rc == 0, f"NV Gamer training exited with code {rc}"

In [ ]:
#@title 📈 ステップ3: マルチゲーム勝率・30×人間味指標・ワールドモデル予測精度を可視化
import json, os
import matplotlib.pyplot as plt

metrics_file = os.path.join(OUT_DIR, "nvgamer_metrics.json")
with open(metrics_file, "r", encoding="utf-8") as f:
    hist = json.load(f)

steps_m = [h["steps"] / 1e6 for h in hist]
wins = [h["win_pct"] for h in hist]
human = [h["human_realism"] for h in hist]
wm_err = [h["world_model_err"] for h in hist]
rews = [h["mean_reward"] for h in hist]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].plot(steps_m, wins, color="#b96bff", lw=2); axes[0].set_title("5-Genre Clear Rate (%)"); axes[0].set_xlabel("Steps (M)"); axes[0].grid(alpha=0.3)
axes[1].plot(steps_m, human, color="#3ee8c8", lw=2); axes[1].set_title("30x Human Realism (/100)"); axes[1].set_xlabel("Steps (M)"); axes[1].grid(alpha=0.3)
axes[2].plot(steps_m, wm_err, color="#ff7aa2", lw=2); axes[2].set_title("World Model Prediction Err"); axes[2].set_xlabel("Steps (M)"); axes[2].grid(alpha=0.3)
axes[3].plot(steps_m, rews, color="#ffa94d", lw=2); axes[3].set_title("Multi-Game Mean Reward"); axes[3].set_xlabel("Steps (M)"); axes[3].grid(alpha=0.3)
plt.tight_layout()
plt.show()
last = hist[-1]
print(f"最新成績: {last['steps']:,} steps | {last['episodes']:,} エピソード | 勝率 {last['win_pct']:.1f}% | 人間味 {last['human_realism']:.1f}/100 | ワールドモデル誤差 {last['world_model_err']:.4f}")

In [ ]:
#@title 💾 ステップ4: 汎用 NV Gamer ウェイト (nvgamer-weights.json) をダウンロード
import os
weights_json = os.path.join(OUT_DIR, "nvgamer-weights.json")
assert os.path.exists(weights_json), "nvgamer-weights.json が見つかりません"
size_kb = os.path.getsize(weights_json) / 1024
print(f"✅ NV Gamer 汎用ウェイト出力完了: {weights_json} ({size_kb:.1f} KB)")
print("👉 Result ロビーの「NV Gamer読込」ボタン、または他の自作ゲームの loadNVGamerWeights() にそのまま読み込めます！")
try:
    from google.colab import files
    files.download(weights_json)
except Exception:
    print("ローカル実行環境のため自動ダウンロードをスキップしました。")